# Morocco Dataset Notebook

This notebook downloads the Morocco crop dataset files, creates one raw merged table, reloads that table, and then prepares a selected crop-plus-background Sentinel-2 time-series dataset. Minor or unreviewed crop labels are removed here; broader `Other` grouping can happen in a later modeling notebook.


## Dependencies

This first cell installs the small package needed to read the shapefile DBF attribute table in Colab.


In [ ]:
%pip install -q dbfread


## Download The Raw Figshare Files

This cell downloads only the two small dataset files needed for this workflow: the shapefile labels and the Sentinel-2/NDVI parcel time series. The large photo archive is intentionally skipped.

In [ ]:
import requests
from pathlib import Path

ARTICLE_ID = 28486022
FILE_IDS = {
    56267117,  # Shapefile_Crop and irrigation dataset.rar
    56267120,  # Sentinel2 bands and Ndvi for surveyed parcels.rar
}

DATA_DIR = Path("data/morocco_crop_irrigation")
DATA_DIR.mkdir(parents=True, exist_ok=True)

files = requests.get(f"https://api.figshare.com/v2/articles/{ARTICLE_ID}/files").json()

for file_info in files:
    if file_info["id"] not in FILE_IDS:
        continue

    out_path = DATA_DIR / file_info["name"]
    print(f"Downloading: {file_info['name']}")

    with requests.get(file_info["download_url"], stream=True) as response:
        response.raise_for_status()
        with out_path.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024 * 8):
                if chunk:
                    output.write(chunk)

    print(f"Saved: {out_path}")


## Extract The Raw Files

This cell extracts the downloaded raw archives into the same working folder. It is intentionally written as simple shell commands for a Colab-style run.

In [ ]:
!mkdir -p "data/morocco_crop_irrigation/"
!tar -xf "data/morocco_crop_irrigation/Shapefile_Crop and irrigation dataset.rar" -C "data/morocco_crop_irrigation/"
!tar -xf "data/morocco_crop_irrigation/Sentinel2 bands and Ndvi for surveyed parcels.rar" -C "data/morocco_crop_irrigation/"


## Load The Raw Dataset Into Memory

This cell reads the raw shapefile attributes and all raw Sentinel-2 parcel CSV files, then joins them by `parcel_id` in memory. It does not clean, filter, or save anything yet.


In [12]:
from pathlib import Path

import pandas as pd
from dbfread import DBF

DATA_DIR = Path("data/morocco_crop_irrigation")
SHAPE_DIR = DATA_DIR / "Shapefile_Crop and irrigation dataset"
S2_DIR = DATA_DIR / "Sentinel2 bands for all surveyed parcels"

In [ ]:


raw_attributes = pd.DataFrame(
    iter(DBF(str(SHAPE_DIR / "lulc_fielddata.dbf"), encoding="utf-8"))
)

frames = []
for csv_path in sorted(S2_DIR.rglob("*_S2_timeseries.csv")):
    if ".ipynb_checkpoints" in csv_path.parts:
        continue

    parcel_id = csv_path.name.removesuffix("_S2_timeseries.csv")
    part = pd.read_csv(csv_path)
    part.insert(0, "parcel_id", parcel_id)
    part.insert(1, "raw_folder", csv_path.parent.name)
    part.insert(2, "raw_file", str(csv_path.relative_to(S2_DIR)))
    frames.append(part)

raw_sentinel2 = pd.concat(frames, ignore_index=True)
raw_merged = raw_sentinel2.merge(
    raw_attributes,
    on="parcel_id",
    how="left",
    suffixes=("_s2", "_label"),
)

print("Raw shapefile attributes:", raw_attributes.shape)
print("Raw Sentinel-2 time series:", raw_sentinel2.shape)
print("Raw merged table:", raw_merged.shape)


In [13]:
# Save the raw merged data to a parquet file for future use
# raw_merged.to_parquet(DATA_DIR / "raw_merged.parquet", index=False)
# load the raw merged data from the parquet file
raw_merged = pd.read_parquet(DATA_DIR / "raw_merged.parquet")

## Explore The Raw Merged Table

From this point onward, the notebook works only with `raw_merged`. This keeps the workflow simple: the raw shapefile attributes and Sentinel-2 time series were already merged and saved as one file.

In [14]:
raw_parcels = raw_merged.drop_duplicates("parcel_id").copy()

print("Raw merged shape:", raw_merged.shape)
print("Unique parcels:", raw_merged["parcel_id"].nunique())
print("Columns:")
print(raw_merged.columns.tolist())

print("\nRaw parcel categories:")
print(raw_parcels["crop"].value_counts(dropna=False))

print("\nTop seasonal crop labels:")
print(raw_parcels["season"].value_counts(dropna=False).head(25))

print("\nIrrigation values:")
print(raw_parcels["irrigation"].value_counts(dropna=False))

print("\nSurvey zones:")
print(raw_parcels["zone"].value_counts(dropna=False))

Raw merged shape: (518907, 29)
Unique parcels: 9993
Columns:
['parcel_id', 'raw_folder', 'raw_file', 'B1', 'B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'NDVI', 'date_s2', 'date_label', 'crop', 'season', 'tree', 'irrigation', 'btw_lines', 'btw_trees', 'height', 'diameter', 'cp_height', 'zone', 'photo']

Raw parcel categories:
crop
Seasonal Crop       6890
Trees               1425
Bare soil            942
Trees + Seasonal     374
Weed                 346
Water                 16
Name: count, dtype: int64

Top seasonal crop labels:
season
Wheat          2768
               2716
Corn           1268
Alfalfa         912
Potatoes        441
Beets           425
Peas            252
Peanut          180
Onions          176
Tomatoes        128
Sugarcane        94
Pumpkins         92
Fava Beans       54
Melons           51
Watermelon       51
Pepper           47
Eggplant         43
Cauliflower      38
Cabbage          33
Parsley          32
Zucchini         26
Carrots       

## Keep Reviewed Crops And Background Parcels

For this cleaned dataset, we keep the three main target crops (`Wheat`, `Corn`, `Alfalfa`), ten reviewed seasonal crops, and the three background classes (`Bare soil`, `Water`, `Weed`). The ten reviewed crops are kept because they have enough samples for this version (`count >= 25`) and are agriculturally close to one of the three main target crops, so they are useful candidates for a later `Other` group instead of being discarded immediately.

The reviewed crop relationships are:

| Future group direction | Reviewed crops kept for later |
|---|---|
| Close to `Wheat` | `Beets`, `Potatoes`, `Peas`, `Onions`, `Fava Beans` |
| Close to `Corn` | `Peanut`, `Tomatoes`, `Sugarcane`, `Rice` |
| Close to `Alfalfa` | `Peas`, `Peanut`, `Fava Beans`, `Parsley` |

Some crops can support more than one future group, such as `Peas`, `Peanut`, and `Fava Beans`. We do not group them here. They stay as original labels so a later modeling notebook can decide the final `Other` strategy.

Tree-only and mixed tree-seasonal parcels are removed because their satellite signal is not a clean seasonal field-crop signal.


In [15]:
main_crops = ["Wheat", "Corn", "Alfalfa"]

other_candidate_crops = [
    "Beets",
    "Potatoes",
    "Peas",
    "Peanut",
    "Onions",
    "Tomatoes",
    "Sugarcane",
    "Fava Beans",
    "Parsley",
    "Rice",
]

background_classes = ["Bare soil", "Water", "Weed"]
kept_seasonal_crops = main_crops + other_candidate_crops

crop_similarity_groups = {
    "Wheat": ["Barley", "Fava Beans", "Peas", "Green Peas", "Beets", "Onions", "Potatoes"],
    "Corn": ["Sorghum", "Sugarcane", "Rice", "Sunflower", "Peanut", "Beans", "Tomatoes"],
    "Alfalfa": ["Fava Beans", "Peas", "Green Peas", "Beans", "Peanut", "Mint", "Parsley"],
}

selected_merged = raw_merged[
    (
        (raw_merged["crop"] == "Seasonal Crop")
        & (raw_merged["season"].isin(kept_seasonal_crops))
    )
    | (raw_merged["crop"].isin(background_classes))
].copy()

selected_merged["label"] = selected_merged.apply(
    lambda row: row["season"] if row["crop"] == "Seasonal Crop" else row["crop"],
    axis=1,
)

selected_merged = selected_merged.rename(
    columns={
        "date_s2": "satellite_date",
        "date_label": "survey_date",
        "zone": "survey_zone",
        "NDVI": "ndvi",
    }
)

selected_merged["irrigation_type"] = selected_merged["irrigation"].astype(str).str.strip()
selected_merged["irrigation_type"] = selected_merged["irrigation_type"].replace(
    {
        "rainfed": "Rainfed",
        "Flood IrrigationFlood Irrigation": "Flood Irrigation",
        "": None,
        "nan": None,
        "None": None,
    }
)

selected_parcel_labels = selected_merged[["parcel_id", "label"]].drop_duplicates()

print("Main crops:", main_crops)
print("Future Other candidate crops:", other_candidate_crops)
print("Background classes:", background_classes)
print("Selected crop/background time-series shape:", selected_merged.shape)
print("Selected parcels:", selected_merged["parcel_id"].nunique())
print("\nSelected label counts:")
print(selected_parcel_labels["label"].value_counts(dropna=False))


Main crops: ['Wheat', 'Corn', 'Alfalfa']
Future Other candidate crops: ['Beets', 'Potatoes', 'Peas', 'Peanut', 'Onions', 'Tomatoes', 'Sugarcane', 'Fava Beans', 'Parsley', 'Rice']
Background classes: ['Bare soil', 'Water', 'Weed']
Selected crop/background time-series shape: (390766, 31)
Selected parcels: 7734

Selected label counts:
label
Wheat         2614
Corn          1252
Bare soil      942
Alfalfa        857
Beets          424
Potatoes       416
Weed           346
Peas           229
Peanut         167
Onions         149
Tomatoes       128
Sugarcane       94
Fava Beans      44
Parsley         31
Rice            25
Water           16
Name: count, dtype: int64


## Compact Dataset Checks

These checks confirm the cleaned crop/background dataset shape, label distribution, irrigation distribution, zone distribution, and Sentinel-2 coverage. The dataset is still time-series data, so one parcel appears on multiple satellite observation dates.


In [16]:
selected_parcels = selected_merged.drop_duplicates("parcel_id")

ts_coverage = (
    selected_merged
    .assign(satellite_date=pd.to_datetime(selected_merged["satellite_date"], errors="coerce"))
    .groupby("parcel_id")
    .agg(
        observation_count=("satellite_date", "count"),
        first_observation=("satellite_date", "min"),
        last_observation=("satellite_date", "max"),
    )
)
ts_coverage["time_span_days"] = (
    ts_coverage["last_observation"] - ts_coverage["first_observation"]
).dt.days

print("Label counts:")
display(selected_parcels["label"].value_counts(dropna=False).to_frame("parcel_count"))

print("Irrigation counts:")
display(selected_parcels["irrigation_type"].value_counts(dropna=False).to_frame("parcel_count"))

print("Zone counts:")
display(selected_parcels["survey_zone"].value_counts(dropna=False).to_frame("parcel_count"))

print("Sentinel-2 coverage summary:")
display(ts_coverage[["observation_count", "time_span_days"]].describe().loc[["min", "25%", "50%", "75%", "max"]])


Label counts:


,parcel_count
label,
Wheat,2614
Corn,1252
Bare soil,942
Alfalfa,857
Beets,424
Potatoes,416
Weed,346
Peas,229
Peanut,167


Irrigation counts:


,parcel_count
irrigation_type,
Flood Irrigation,2461
Drip Irrigation,1391
None,1274
Sprinkler Irrigation,1215
Rainfed,1107
Basin Irrigation,254
Pivot Irrigation,32


Zone counts:


,parcel_count
survey_zone,
Doukkala,3948
Gharb,1292
Tadla,1161
el haouz,918
Statte,248
Souss,167


Sentinel-2 coverage summary:


,observation_count,time_span_days
min,18.0,305.0
25%,33.0,330.0
50%,35.0,353.0
75%,65.0,358.0
max,301.0,360.0


## Coverage And Cleaning Conclusion

The cleaned output keeps reviewed seasonal crop labels plus `Bare soil`, `Water`, and `Weed` as background classes. It includes the three main crops (`Wheat`, `Corn`, `Alfalfa`) and ten reviewed crops that can later be considered for an `Other` class.

The Sentinel-2 coverage is broad across 2024, but the number of observations per parcel is uneven, so later modeling should use aggregation, time-window features, interpolation, or a sequence model that can handle variable-length inputs.

The output does not keep `source_folder`, `source_file`, raw irrigation text, photo paths, tree labels, mixed tree-seasonal labels, tree-only labels, unreviewed rare crops, or tree-structure measurement columns.


## Save One Selected Crop/Background Time-Series File

This checkpoint saves one easy-load Parquet file for the next notebook. The file is still a time-series table: one row per parcel per Sentinel-2 observation date.


In [17]:
final_columns = [
    "parcel_id",
    "satellite_date",
    "B1",
    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B9",
    "B11",
    "B12",
    "ndvi",
    "survey_date",
    "label",
    "irrigation_type",
    "survey_zone",
]

final_dataset = selected_merged[final_columns].copy()
output_path = DATA_DIR / "morocco_selected_crop_background_timeseries.parquet"
final_dataset.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Final selected crop/background time-series shape:", final_dataset.shape)
print("Final parcels:", final_dataset["parcel_id"].nunique())
print("Final columns:")
print(final_dataset.columns.tolist())


Saved: data\morocco_crop_irrigation\morocco_selected_crop_background_timeseries.parquet
Final selected crop/background time-series shape: (390766, 19)
Final parcels: 7734
Final columns:
['parcel_id', 'satellite_date', 'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'ndvi', 'survey_date', 'label', 'irrigation_type', 'survey_zone']
